# 13 · Explainability, Error Analysis and Fairness (Dev)

**What this notebook covers**

- Permutation importance and agreement with Logistic Regression
- SHAP global and local explanations
- Error analysis by segment and most confident mistakes
- Fairness checks by market segment and country

## 1. Environment setup
### 1.1 Imports
`shap` is optional. If it is not installed, the SHAP sections are skipped and
permutation importance (always available) carries the explanation.

In [ ]:
import hashlib
import json
import platform
import sys
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit
from sklearn.inspection import permutation_importance
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
print(f"SHAP available: {SHAP_AVAILABLE}")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

### 1.2 Configuration

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_PREFIX = "13"
PERMUTATION_SAMPLE = 5000
PERMUTATION_REPEATS = 5
SHAP_SAMPLE = 2000
TOP_N_COUNTRIES = 10
FAIRNESS_MIN_GROUP_SIZE = 200
FAIRNESS_MIN_POSITIVES = 30
TPR_GAP_FLAG = 0.10            # |group TPR - overall TPR| above this is flagged
FPR_RATIO_BOUNDS = (0.8, 1.25)  # group FPR / overall FPR outside this range is flagged ("four-fifths"-style rule)

### 1.3 Project paths, shared helpers and plotting style

In [ ]:
# Resolve the repository root regardless of where the kernel was started
# (VS Code starts kernels in notebooks/, Jupyter may start them at the repo root).
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "DECISIONS.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (no DECISIONS.md in any parent directory).")


PROJECT_ROOT = find_project_root()

PATHS = {
    "raw": PROJECT_ROOT / "data" / "hotel_bookings.csv",
    "clean": PROJECT_ROOT / "data" / "interim" / "hotel_bookings_clean.csv",
    "splits": PROJECT_ROOT / "data" / "processed" / "splits",
    "features": PROJECT_ROOT / "data" / "processed" / "features",
    "artifacts": PROJECT_ROOT / "artifacts",
    "preprocessors": PROJECT_ROOT / "artifacts" / "preprocessors",
    "model_outputs": PROJECT_ROOT / "artifacts" / "model_outputs",
    "models": PROJECT_ROOT / "models",
    "figures": PROJECT_ROOT / "reports" / "figures",
    "tables": PROJECT_ROOT / "reports" / "tables",
    "notebooks": PROJECT_ROOT / "notebooks",
}
for key in ("artifacts", "preprocessors", "model_outputs", "models", "figures", "tables"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Columns that must always be treated as categorical strings when re-reading CSVs.
STRING_COLUMNS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment", "distribution_channel",
    "reserved_room_type", "assigned_room_type", "deposit_type", "agent", "company",
    "customer_type", "arrival_season",
]
TARGET = "is_canceled"
ID_COLUMN = "booking_id"
LEAKAGE_COLUMNS = ["reservation_status", "reservation_status_date"]

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Streaming SHA-256 of a file (used to fingerprint every artifact)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact missing: {path.relative_to(PROJECT_ROOT)}")
    return json.loads(path.read_text())


def read_frame(path: Path) -> pd.DataFrame:
    """Read a project CSV with categorical columns forced to plain Python strings.

    `keep_default_na=False` + an explicit NA list stops pandas from turning legitimate
    category labels into NaN; numeric columns are parsed normally.
    """
    if not path.exists():
        raise FileNotFoundError(f"Required data file missing: {path.relative_to(PROJECT_ROOT)}")
    frame = pd.read_csv(
        path,
        dtype={col: object for col in STRING_COLUMNS},
        keep_default_na=False,
        na_values=["", "NaN", "nan"],
    )
    present = [c for c in STRING_COLUMNS if c in frame.columns]
    frame[present] = frame[present].astype(object)
    return frame

In [ ]:
# One visual language for every figure in the project:
#  - blue = not cancelled, orange = cancelled (validated colour-blind-safe pair)
#  - single-series charts use blue; recessive grey grid; no top/right spines; no dual axes.
COLORS = {
    "not_canceled": "#2a78d6",
    "canceled": "#eb6834",
    "neutral": "#8a8985",
    "ink": "#0b0b0b",
    "ink_muted": "#52514e",
}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SEQUENTIAL_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)
DIVERGING_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "project_diverging", ["#2a78d6", "#e8e7e3", "#eb6834"]
)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "axes.prop_cycle": mpl.cycler(color=SERIES_COLORS),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae",
    "axes.labelcolor": COLORS["ink_muted"],
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": "#e8e7e3",
    "grid.linewidth": 0.8,
    "xtick.color": COLORS["ink_muted"],
    "ytick.color": COLORS["ink_muted"],
    "lines.linewidth": 2,
    "legend.frameon": False,
})


def save_figure(fig, name: str) -> Path:
    """Save a figure under reports/figures with this notebook's numeric prefix."""
    path = PATHS["figures"] / f"{NOTEBOOK_PREFIX}_{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"Saved figure -> {path.relative_to(PROJECT_ROOT)}")
    return path

### 1.4 Calibration helpers (same definitions as notebook 12)

In [ ]:
EPS = 1e-6


def fit_calibrator(method: str, p_raw, y):
    """Fit a probability-level calibrator. 'none' returns None (identity)."""
    p_raw = np.clip(np.asarray(p_raw, dtype=float), EPS, 1 - EPS)
    if method == "none":
        return None
    if method == "sigmoid":          # Platt scaling on the logit of the raw score
        return LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_raw).reshape(-1, 1), y)
    if method == "isotonic":
        return IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(p_raw, y)
    raise ValueError(method)


def apply_calibration(method: str, calibrator, p_raw):
    p_raw = np.clip(np.asarray(p_raw, dtype=float), EPS, 1 - EPS)
    if method == "none":
        return p_raw
    if method == "sigmoid":
        return calibrator.predict_proba(logit(p_raw).reshape(-1, 1))[:, 1]
    return np.clip(calibrator.predict(p_raw), 0.0, 1.0)

## 2. Load the locked policy, the final model and Dev
The bundle hash must match the policy, so we explain exactly the model that will be
tested. Re-predicted Dev scores must also equal the scores logged by the model
notebook (a reproducibility check).

In [ ]:
policy = load_json(PATHS["artifacts"] / "decision_policy.json")
bundle_path = PROJECT_ROOT / policy["bundle_path"]
assert sha256_file(bundle_path) == policy["bundle_sha256"], "Final model bundle changed after the policy was locked."
bundle = joblib.load(bundle_path)
model, MODEL_ID = bundle["model"], policy["model_id"]
THRESHOLD = policy["operating_threshold"]

dev = read_frame(PATHS["features"] / "dev.csv")
X_dev, y_dev = dev.drop(columns=[TARGET, ID_COLUMN]), dev[TARGET].astype(int).to_numpy()

p_raw = model.predict_proba(X_dev)[:, 1]
logged = pd.read_csv(PATHS["model_outputs"] / MODEL_ID / "dev_predictions.csv")
assert np.allclose(p_raw, logged["y_proba"].to_numpy(), atol=1e-6), "Re-predicted Dev scores differ from the logged ones."

p_cal = apply_calibration(bundle["calibration_method"], bundle["calibrator"], p_raw)
y_pred = (p_cal >= THRESHOLD).astype(int)
print(f"Model {MODEL_ID} | threshold {THRESHOLD:.2f} | Dev flag rate {y_pred.mean():.2%}")

## 3. Global explanations
### 3.1 Permutation importance (selected model, raw input columns)
Permutation importance is the drop in Dev PR-AUC when one original column is shuffled.
It works for any model and permutes a column's one-hot children together.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
perm_idx = rng.choice(len(X_dev), size=min(PERMUTATION_SAMPLE, len(X_dev)), replace=False)
used_columns = sorted({c for name, _, cols in model.named_steps["preprocess"].transformers_ if name != "remainder" for c in cols})


def permutation_table(estimator) -> pd.DataFrame:
    """Permute every input column (the pipeline expects the full frame) and report the consumed ones."""
    result = permutation_importance(estimator, X_dev.iloc[perm_idx], y_dev[perm_idx],
                                    scoring="average_precision", n_repeats=PERMUTATION_REPEATS,
                                    random_state=RANDOM_STATE, n_jobs=-1)
    table = pd.DataFrame({"feature": X_dev.columns, "mean_drop_pr_auc": result.importances_mean, "std": result.importances_std})
    return (table[table["feature"].isin(used_columns)]
            .sort_values("mean_drop_pr_auc", ascending=False).reset_index(drop=True))


perm_selected = permutation_table(model)
perm_selected.to_csv(PATHS["tables"] / "13_permutation_importance_selected.csv", index=False)

top = perm_selected.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["feature"], top["mean_drop_pr_auc"], xerr=top["std"], color=COLORS["not_canceled"], height=0.65,
        error_kw={"ecolor": COLORS["ink_muted"], "elinewidth": 1, "capsize": 2})
ax.axvline(0, color=COLORS["ink_muted"], linewidth=1)
ax.set_xlabel("Drop in Dev PR-AUC when shuffled")
ax.set_title(f"What the selected model relies on ({MODEL_ID})", loc="left")
ax.grid(axis="y", visible=False)
save_figure(fig, "permutation_importance_selected")
plt.show()

### 3.2 Cross-model agreement: selected model vs Logistic Regression
If an interpretable linear model and the selected model rank the same drivers on top,
the explanation is more trustworthy. Large disagreements point to interactions or
non-linearities the linear model cannot capture.

In [ ]:
logreg_path = PATHS["models"] / "logreg__day_zero.joblib"
if MODEL_ID != "logreg__day_zero" and logreg_path.exists():
    perm_logreg = permutation_table(joblib.load(logreg_path))
    agreement = (perm_selected.assign(rank_selected=lambda d: d.index + 1)[["feature", "rank_selected", "mean_drop_pr_auc"]]
                 .merge(perm_logreg.assign(rank_logreg=lambda d: d.index + 1)[["feature", "rank_logreg"]], on="feature"))
    spearman = agreement[["rank_selected", "rank_logreg"]].corr(method="spearman").iloc[0, 1]
    print(f"Spearman rank agreement (selected vs logistic regression): {spearman:.2f}")
    agreement.to_csv(PATHS["tables"] / "13_importance_agreement.csv", index=False)
    display(agreement.head(15))
else:
    print("Selected model is the logistic regression (or it is unavailable); agreement check not applicable.")

### 3.3 SHAP global summary
SHAP attributes each prediction to the encoded features, consistently across rows.
`TreeExplainer` is exact and fast for tree ensembles, and `LinearExplainer` handles
linear models. Kernel SHAP for the SVM would be too slow, so permutation importance
covers it.

In [ ]:
model_key = MODEL_ID.split("__")[0]
estimator = model.named_steps["model"]
preprocessor = model.named_steps["preprocess"]
feature_names = preprocessor.get_feature_names_out()
shap_idx = rng.choice(len(X_dev), size=min(SHAP_SAMPLE, len(X_dev)), replace=False)
X_dev_encoded = pd.DataFrame(preprocessor.transform(X_dev.iloc[shap_idx]), columns=feature_names)

shap_values = None
if SHAP_AVAILABLE and model_key in {"decision_tree", "random_forest", "xgboost"}:
    shap_values = shap.TreeExplainer(estimator)(X_dev_encoded, check_additivity=False)
elif SHAP_AVAILABLE and model_key == "logreg":
    shap_values = shap.LinearExplainer(estimator, shap.maskers.Independent(X_dev_encoded, max_samples=500))(X_dev_encoded)
if shap_values is not None and shap_values.values.ndim == 3:   # (rows, features, classes) -> positive class
    shap_values = shap_values[..., 1]
print("SHAP computed." if shap_values is not None else "SHAP skipped for this model type (see permutation importance).")

### 3.4 SHAP bar and beeswarm plots
The bar shows mean |SHAP| (overall impact). The beeswarm shows direction: each dot is a
booking, coloured by the feature's value from low (light) to high (dark).

In [ ]:
if shap_values is not None:
    shap.plots.bar(shap_values, max_display=15, show=False)
    plt.title("Mean |SHAP|: top 15 encoded features", loc="left")
    save_figure(plt.gcf(), "shap_bar")
    plt.show()

    shap.plots.beeswarm(shap_values, max_display=15, color=SEQUENTIAL_CMAP, show=False)
    plt.title("SHAP beeswarm: direction and spread of effects", loc="left")
    save_figure(plt.gcf(), "shap_beeswarm")
    plt.show()

    mean_abs = pd.Series(np.abs(shap_values.values).mean(axis=0), index=feature_names, name="mean_abs_shap")
    mean_abs.sort_values(ascending=False).to_csv(PATHS["tables"] / "13_shap_mean_abs.csv")

## 4. Local explanations: why was a booking flagged?
These are the per-booking explanations front-office staff should see next to a score
(docs/07 §7.4, transparency). Shown for the highest-risk booking and for a borderline
booking just above the threshold.

In [ ]:
if shap_values is not None:
    sample_scores = p_cal[shap_idx]
    examples = {"highest risk": int(np.argmax(sample_scores))}
    above = np.where(sample_scores >= THRESHOLD)[0]
    if len(above):
        examples["borderline (just above threshold)"] = int(above[np.argmin(sample_scores[above])])
    for label, i in examples.items():
        shap.plots.waterfall(shap_values[i], max_display=10, show=False)
        plt.title(f"{label}: calibrated risk {sample_scores[i]:.2f} (booking {dev[ID_COLUMN].iloc[shap_idx[i]]})", loc="left")
        save_figure(plt.gcf(), f"shap_waterfall_{label.split()[0]}")
        plt.show()

## 5. Error analysis at the locked threshold
### 5.1 Error profile by segment
For each segment: prevalence, flag rate, precision, recall (TPR), false-negative rate
(missed cancellations) and false-positive rate (unneeded contacts). This tells the
business *where* the model is blind (docs/06 §6.6).

In [ ]:
LEAD_TIME_BINS = [-1, 7, 30, 90, 180, 365, np.inf]
LEAD_TIME_LABELS = ["0-7d", "8-30d", "31-90d", "91-180d", "181-365d", ">365d"]
analysis = dev.assign(y_true=y_dev, y_pred=y_pred, p_cal=p_cal,
                      lead_time_band=pd.cut(dev["lead_time"], LEAD_TIME_BINS, labels=LEAD_TIME_LABELS).astype(str))


def group_rates(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    def rates(g):
        tp = ((g.y_pred == 1) & (g.y_true == 1)).sum(); fp = ((g.y_pred == 1) & (g.y_true == 0)).sum()
        fn = ((g.y_pred == 0) & (g.y_true == 1)).sum(); tn = ((g.y_pred == 0) & (g.y_true == 0)).sum()
        return pd.Series({
            "n": len(g), "positives": int(g.y_true.sum()), "prevalence": g.y_true.mean(), "flag_rate": g.y_pred.mean(),
            "precision": tp / (tp + fp) if tp + fp else np.nan, "tpr_recall": tp / (tp + fn) if tp + fn else np.nan,
            "fnr": fn / (tp + fn) if tp + fn else np.nan, "fpr": fp / (fp + tn) if fp + tn else np.nan,
        })
    return pd.DataFrame({group: rates(g) for group, g in frame.groupby(column, observed=True)}).T.sort_values("n", ascending=False)


SEGMENTS = ["hotel", "market_segment", "deposit_type", "customer_type", "distribution_channel", "lead_time_band"]
error_profiles = {segment: group_rates(analysis, segment) for segment in SEGMENTS}
pd.concat(error_profiles, names=["segment", "group"]).to_csv(PATHS["tables"] / "13_error_profile_by_segment.csv")
error_profiles["market_segment"].round(3)

### 5.2 Missed cancellations vs unneeded contacts by market segment
Dot plot with two series on one shared 0–1 axis. Groups under 100 bookings are hidden.

In [ ]:
profile = error_profiles["market_segment"].query("n >= 100").sort_values("fnr")
fig, ax = plt.subplots(figsize=(9, 0.5 * len(profile) + 1.8))
y_positions = np.arange(len(profile))
ax.hlines(y_positions, profile[["fnr", "fpr"]].min(axis=1), profile[["fnr", "fpr"]].max(axis=1), color="#e8e7e3", linewidth=3)
ax.scatter(profile["fnr"], y_positions, s=80, color=SERIES_COLORS[1], edgecolor="white", linewidth=2, label="Missed cancellations (FNR)", zorder=3)
ax.scatter(profile["fpr"], y_positions, s=80, color=SERIES_COLORS[0], edgecolor="white", linewidth=2, label="Unneeded contacts (FPR)", zorder=3)
ax.set_yticks(y_positions, profile.index)
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_xlim(0, 1)
ax.set_title("Where the model errs, by market segment (Dev)", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
save_figure(fig, "error_profile_market_segment")
plt.show()

### 5.3 Most confident mistakes
The confident false negatives (very low score, yet cancelled) and confident false
positives show the patterns the model cannot see. They are a starting point for new
features or data collection.

In [ ]:
show_cols = [ID_COLUMN, "hotel", "market_segment", "deposit_type", "customer_type", "lead_time", "previous_cancellations",
             "total_of_special_requests", "p_cal"]
confident_fn = analysis[(analysis.y_true == 1) & (analysis.y_pred == 0)].nsmallest(10, "p_cal")[show_cols]
confident_fp = analysis[(analysis.y_true == 0) & (analysis.y_pred == 1)].nlargest(10, "p_cal")[show_cols]
pd.concat({"false_negative": confident_fn, "false_positive": confident_fp}).to_csv(PATHS["tables"] / "13_confident_errors.csv")
display(confident_fn)
display(confident_fp)

### 5.4 Does the Non-Refund anomaly drive false positives?
docs/06 §6.6 asks for this explicitly: if FPs cluster in `deposit_type = Non Refund`,
the overall metric is not equally reliable across segments.

In [ ]:
fp_mask = (analysis.y_true == 0) & (analysis.y_pred == 1)
fp_share = analysis.loc[fp_mask, "deposit_type"].value_counts(normalize=True).rename("share_of_false_positives")
base_share = analysis["deposit_type"].value_counts(normalize=True).rename("share_of_bookings")
pd.concat([fp_share, base_share], axis=1).fillna(0).round(4)

## 6. Fairness audit
### 6.1 Group metrics with minimum-support rules
`country` correlates with nationality and `market_segment` with booking channel. A risk
score must not quietly treat groups very differently (docs/07 §7.4). For each group we
compare against overall:

- **Selection rate** (share flagged): the demographic-parity view.
- **TPR gap**: are cancellations caught equally? (equal opportunity)
- **FPR ratio**: are honouring guests contacted equally often? (equalised odds)

Groups below the minimum size or positive count are reported as *insufficient data*
rather than over-interpreted.

In [ ]:
top_countries = analysis["country"].value_counts().head(TOP_N_COUNTRIES).index
analysis["country_group"] = analysis["country"].where(analysis["country"].isin(top_countries), "Other")
overall = group_rates(analysis.assign(all="All"), "all").iloc[0]


def fairness_table(column: str) -> pd.DataFrame:
    table = group_rates(analysis, column)
    table["selection_rate"] = table["flag_rate"]
    table["tpr_gap"] = table["tpr_recall"] - overall["tpr_recall"]
    table["fpr_ratio"] = table["fpr"] / overall["fpr"]
    sufficient = (table["n"] >= FAIRNESS_MIN_GROUP_SIZE) & (table["positives"] >= FAIRNESS_MIN_POSITIVES)
    table["status"] = np.where(~sufficient, "insufficient data",
                      np.where((table["tpr_gap"].abs() > TPR_GAP_FLAG) |
                               ~table["fpr_ratio"].between(*FPR_RATIO_BOUNDS), "FLAG", "ok"))
    return table[["n", "positives", "prevalence", "selection_rate", "tpr_recall", "tpr_gap", "fpr", "fpr_ratio", "precision", "status"]]


fairness = {"country_group": fairness_table("country_group"), "market_segment": fairness_table("market_segment")}
pd.concat(fairness, names=["attribute", "group"]).to_csv(PATHS["tables"] / "13_fairness_audit.csv")
print(f"Overall: TPR {overall['tpr_recall']:.3f}, FPR {overall['fpr']:.3f}, selection rate {overall['flag_rate']:.3f}")
fairness["country_group"].round(3)

### 6.2 Market-segment fairness table

In [ ]:
fairness["market_segment"].round(3)

### 6.3 TPR and FPR by country group
Two panels on a shared 0–1 scale. The dashed line is the overall rate. Groups with
insufficient data are left out of the chart.

In [ ]:
table = fairness["country_group"].query("status != 'insufficient data'").sort_values("tpr_recall")
fig, axes = plt.subplots(1, 2, figsize=(14, 0.45 * len(table) + 2), sharey=True)
for ax, metric, title, color in [(axes[0], "tpr_recall", "Cancellations caught (TPR)", SERIES_COLORS[0]),
                                 (axes[1], "fpr", "Honouring guests flagged (FPR)", SERIES_COLORS[1])]:
    ax.scatter(table[metric], table.index, s=80, color=color, edgecolor="white", linewidth=2, zorder=3)
    ax.axvline(overall[metric], color=COLORS["neutral"], linestyle="--", linewidth=1.2, label=f"Overall {overall[metric]:.0%}")
    for name, value, status in zip(table.index, table[metric], table["status"]):
        if status == "FLAG":
            ax.annotate("flag", (value, name), textcoords="offset points", xytext=(8, -3), fontsize=8, color=COLORS["ink_muted"])
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_xlim(0, 1)
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="lower right")
fig.suptitle("Error-rate parity across country groups (Dev)", x=0.01, ha="left", fontweight="semibold")
fig.tight_layout()
save_figure(fig, "fairness_country")
plt.show()

## 7. Findings to carry into the report (fill in after running)
- **Top drivers** (§3): list the top 5 and whether linear and non-linear models agree.
- **Blind spots** (§5): segments with the highest FNR. Say whether FPs concentrate in Non-Refund deposits.
- **Fairness** (§6): every `FLAG` row with a plausible explanation (e.g. the PRT
  country-recording hypothesis from notebook 03) and a mitigation (human review, no
  automatic action, per-group monitoring).
- Update `docs/07-recommendation-limitations.md` §7.3–7.4 with these numbers.